In [1]:
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import random

from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeManilaV2,FakeNairobiV2,FakeMelbourneV2,FakeGuadalupeV2
import stim
from qiskit_optimization.applications import Knapsack
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.circuit.library import QAOAAnsatz


import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, generate_qiskit_param_map
from testing_scripts.graphs_utils import generate_random_complete_graph,generate_k_regular_graph
from testing_scripts.energy_utils import evaluate_energy

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [2]:
prob = Knapsack(values=[1, 2, 3], weights=[1, 2, 3], max_weight=4)
qp = prob.to_quadratic_program()
print(qp.prettyprint())

# intermediate QUBO form of the optimization problem
conv = QuadraticProgramToQubo()
qubo = conv.convert(qp)

# qubit Hamiltonian and offset
op, offset = qubo.to_ising()
print(f"num qubits: {op.num_qubits}, offset: {offset}\n")
print(op)

cost_hamiltonian = op
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real
reversed_paulis = [p[::-1] for p in paulis]

Problem name: Knapsack

Maximize
  x_0 + 2*x_1 + 3*x_2

Subject to
  Linear constraints (1)
    x_0 + 2*x_1 + 3*x_2 <= 4  'c0'

  Binary variables (3)
    x_0 x_1 x_2

num qubits: 6, offset: 39.0

SparsePauliOp(['IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[ -6.5+0.j, -13. +0.j, -19.5+0.j,  -7. +0.j, -14. +0.j,  -7. +0.j,
   7. +0.j,  10.5+0.j,   3.5+0.j,   7. +0.j,   3.5+0.j,  21. +0.j,
   7. +0.j,  14. +0.j,   7. +0.j,  10.5+0.j,  21. +0.j,  10.5+0.j,
   7. +0.j,   3.5+0.j,   7. +0.j])


In [3]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=1)

dec_circ = circuit.decompose().decompose()
dag = circuit_to_dag(dec_circ)


In [ ]:
def parameter_renaming(dec_circ):
    dag = circuit_to_dag(dec_circ)
    gamma_counter, beta_counter = 0, 0
    angle_multipliers = {}
    for node in dag.op_nodes():
        if node.op.params and isinstance(node.op.params[0], ParameterExpression):
            param_name = list(node.op.params[0].parameters)[0].name 
            if "β" in param_name:
                
                beta = Parameter(f'beta_{beta_counter}')
                new_params = [beta if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
                new_op = node.op.copy()
                new_op.params = new_params  # Create a modified version of the operation
                dag.substitute_node(node, new_op)  # Replace the node in the DAG
                beta_counter += 1

                multiplier = float(str(node.op.params[0]).split("*")[0])

                angle_multipliers[beta.name] = multiplier
            

            elif "γ" in param_name:
                gamma = Parameter(f'gamma_{gamma_counter}')
                new_params = [gamma if (isinstance(p, ParameterExpression)) else p for p in node.op.params]
                new_op = node.op.copy()
                new_op.params = new_params  # Create a modified version of the operation
                dag.substitute_node(node, new_op)  # Replace the node in the DAG
                gamma_counter += 1

                multiplier = float(str(node.op.params[0]).split("*")[0])

                angle_multipliers[gamma.name] = multiplier

    # Convert DAG back to a circuit
    new_qc = dag_to_circuit(dag)
    return new_qc, circuit_to_dag(new_qc) , angle_multipliers

# Usage
circuit, dag , angle_multipliers= parameter_renaming(dec_circ)

In [5]:
# Transform qiskit circ. to stim.
modified_circ = modify_circuit(circuit)
pcirc = transform_to_allowed_gates(modified_circ)

stim_circ = qiskit_to_stim(pcirc)

stim_circ.stim_circuit().diagram()

q0: -S-SQRT_X-Z-I------------@---@------------@---@------------------@---@------------------------@---@------------------------------@---@-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------------------------------
                             |   |            |   |                  |   |                        |   |                              |   |
q1: ------------S-SQRT_X-S-I-X-I-X------------|---|-@---@------------|---|-@---@------------------|---|-@---@------------------------|---|-----------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------------------------------
                                              |   | |   |            |   | |   |                  |   | |   |                        |   |                 |   |
q2: -----------------------------S-SQRT_X-S-I-X-I-X-X-I-X------------|---|-|---|-@---@------------|---|-|---|-@---@------------------|---|-----------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------------------------------
                                                                     |   | |   | |   |            |   | |   | |   |                  |   |                 |   |                   |   |
q3: ----------------------------------------------------S-SQRT_X-S-I-X-I-X-X-I-X-X-I-X------------|---|-|---|-|---|-@---@------------|---|-----------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------------------------------
                                                                                                  |   | |   | |   | |   |            |   |                 |   |                   |   |                   |   |
q4: ---------------------------------------------------------------------------------S-SQRT_X-S-I-X-I-X-X-I-X-X-I-X-X-I-X------------|---|-----------------|---|-------------------|---|-------------------|---|-------------------@---@-S-SQRT_X-I-SQRT_X-S-------------------
                                                                                                                                     |   |                 |   |                   |   |                   |   |                   |   |
q5: --------------------------------------------------------------------------------------------------------------------S-SQRT_X-S-I-X-I-X-----------------X-I-X-------------------X-I-X-------------------X-I-X-------------------X-I-X-------------------S-SQRT_X-I-SQRT_X-S-

In [6]:
# Parameter Mapping
param_map = generate_qiskit_param_map(pcirc)

In [7]:
stim_circ.define_parameter_map(param_map)

In [8]:
# we can perform CAFQA by using the main optimization function "claptonize"

ks_best, _, energy_best = claptonize(
    reversed_paulis,
    coeffs,
    stim_circ,
    n_proc=4,           # total number of processes in parallel
    n_starts=4,         # number of random genetic algorithm starts in parallel
    n_rounds=1,         # number of budget rounds, if None it will terminate itself
    callback=print,     # callback for internal parameter (#iteration, energies, ks) processing
    budget=10           # budget per genetic algorithm instance
)

STARTING ROUND 0




started GA at id 1 with 1 procs

started GA at id 2 with 1 procs


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


started GA at id None with 1 procs


GA parameters used for this experiment:
  num_generations=10
  num_parents_mating=20
  population_size=100
  num_genes=27
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5GA parameters used for this experiment:
  num_generations=10
  num_parents_mating=20
  population_size=100
  num_genes=27
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5

started GA at id 3 with 1 procs


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


GA parameters used for this experiment:
  num_generations=10
  num_parents_mating=20
  population_size=100
  num_genes=27
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5


/global/homes/d/dhanvib/.conda/envs/qaoa_w_sage/lib/python3.10/site-packages/pygad/pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")


GA parameters used for this experiment:
  num_generations=10
  num_parents_mating=20
  population_size=100
  num_genes=27
  parent_selection_type=tournament
  keep_parents=-1
  crossover_type=single_point
  mutation_type=adaptive
  crossover_probability=0.9
  mutation_probability=(0.25, 0.01)
  keep_elitism=5
[0, array([-28., -14., -14.,   0.]), array([3, 1, 0, 2, 1, 0, 2, 3, 2, 2, 3, 0, 0, 3, 3, 2, 2, 0, 3, 0, 3, 3,
       0, 2, 2, 1, 1], dtype=object)]
[0, array([-28., -14., -14.,   0.]), array([3, 1, 0, 2, 1, 0, 2, 3, 2, 2, 3, 0, 0, 3, 3, 2, 2, 0, 3, 0, 3, 3,
       0, 2, 2, 1, 1], dtype=object)]
[0, array([-28., -14., -14.,   0.]), array([3, 1, 0, 2, 1, 0, 2, 3, 2, 2, 3, 0, 0, 3, 3, 2, 2, 0, 3, 0, 3, 3,
       0, 2, 2, 1, 1], dtype=object)]
[0, array([-28., -14., -14.,   0.]), array([3, 1, 0, 2, 1, 0, 2, 3, 2, 2, 3, 0, 0, 3, 3, 2, 2, 0, 3, 0, 3, 3,
       0, 2, 2, 1, 1], dtype=object)]
[1, array([-28., -14., -14.,   0.]), array([0, 0, 1, 1, 1, 3, 0, 2, 2, 0, 0, 2, 1, 3, 1, 1, 1, 2,

In [9]:
energy_best

np.float64(-35.0)

In [10]:
# ks_best = []
# random.seed(9) #9
# for i in range(pcirc.num_parameters):
#     ks_best.append(random.randint(0, 3))

# ks_best

In [11]:
stim_circ.assign(ks_best)

In [12]:
# Solve with classical Eigensolver for comparison
eigensolver = NumPyMinimumEigensolver()
exact_solution = eigensolver.compute_minimum_eigenvalue(cost_hamiltonian).eigenvalue.real
print("Exact Energy from Eigensolver:", exact_solution)

Exact Energy from Eigensolver: -43.0


In [13]:
ordered_params = [param.name for param in pcirc.parameters]
mults = [(np.pi/2) * (1) for param in ordered_params]
cafqa_angles = [param * (multiplier) for param,multiplier in zip(ks_best,mults)] #This has to be in the order we come across the gates.

In [14]:
energies = [evaluate_energy(pcirc, cost_hamiltonian, cafqa_angles) for _ in range(10)]
average_energy = np.mean(energies)
print(f"Average Energy: {average_energy}")

Average Energy: -6.9864501953125


In [15]:
# Random Initalization 
random_angles = np.random.random(len(ks_best))
random_energies = [evaluate_energy(pcirc, cost_hamiltonian, random_angles) for _ in range(100)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

Minimum Energy found with Random initialization over 100 runs: 21.246337890625


In [17]:
ks_best

[3,
 2,
 2,
 2,
 3,
 0,
 2,
 0,
 3,
 3,
 0,
 3,
 2,
 1,
 3,
 2,
 3,
 2,
 3,
 1,
 3,
 2,
 2,
 2,
 3,
 3,
 3]